<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/02_Fine_Tuning_Kronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM - Deliverable 2--Fine-Tuning Kronos on Indian Equity Data

### Objectives

- Load Kronos Foundation Model
- Load Tokenizer
- Load Processed Dataset
- Fine-tune using LoRA
- Evaluate against Zero-Shot
- Save Best Model



#Setup

In [73]:
!git clone https://github.com/shiyu-coder/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [74]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [3]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import random
import sys
import warnings
import yaml
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

warnings.filterwarnings("ignore")

In [6]:
!pip -q install transformers peft bitsandbytes accelerate huggingface_hub sentencepiece pyyaml safetensors

In [7]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [8]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"

PROCESSED_PATH = os.path.join(PROJECT_ROOT,"data/processed")
MODEL_PATH = os.path.join(PROJECT_ROOT,"models")

#Fine Tuning

In [9]:
from model import Kronos, KronosTokenizer
from train_sequential import SequentialTrainer
from config_loader import CustomFinetuneConfig
from finetune_base_model import create_dataloaders

In [10]:
dataset_path = os.path.join(PROCESSED_PATH,"RELIANCE.NS.csv")
df = pd.read_csv(dataset_path)
df.head()

,Date,Close,High,Low,Open,Volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398


In [11]:
df = df.rename(columns={"Date":"timestamps","Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
df["amount"] = 0  #amt is optional
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.head()

,timestamps,close,high,low,open,volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta,amount
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910,0
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455,0
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226,0
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940,0
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398,0


In [12]:
save_path = "/content/Kronos/finetune_csv/data/prism_train.csv"
df.to_csv(save_path,index=False)
print(save_path)

/content/Kronos/finetune_csv/data/prism_train.csv


In [13]:
config = {

"data":{                     #Dataset ko kaise prepare aur split karna hai
"data_path":save_path,
"lookback_window":30,        #Kitna past data input banana hai
"predict_window":5,
"max_context":30,            #Model maximum kitne tokens dekh sakta hai
"clip":5,                    #Values ko safe range me clip karna (outliers control) like yaha pe matlab hai values ko [-5, +5] range ke andar restrict karna hai
"train_ratio":0.8,
"val_ratio":0.2,
"test_ratio":0.0             # kyuki iss fine-tuning phase me sirf training aur validation use ho rahe hain
},

"training":{                 #Training ka behaviour define karta hai
"tokenizer_epochs":2,
"basemodel_epochs":2,
"batch_size":32,
"learning_rate":1e-4,
"num_workers":2
},

"model_paths":{              #Models kahan se load aur kahan save honge
"exp_name":"PRISM",
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base", #HF se Kronos tokenizer load karo
"pretrained_predictor":"NeoQuasar/Kronos-base",           #HF se pretrained Kronos model load karo.
"base_save_path":"/content/Kronos/checkpoints"            #Fine-tuned checkpoints isi folder me save honge
}
}

In [14]:
config_path = "/content/Kronos/finetune_csv/configs/prism.yaml"
with open(config_path,"w") as f:           #Python dictionary (config) ko prism.yaml file me save kar rahe hain, taaki poora project us configuration ko read kar sake
    yaml.dump(config,f)
cfg = CustomFinetuneConfig(config_path)
cfg.print_config_summary()

Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [15]:
#create_dataloaders(cfg) configuration ke basis par train aur validation DataLoaders banata hai,
#aur *_ ka use karke sirf required loaders ko store kiya jata hai jabki baaki returned values ignore kar di jaati hain.
train_loader, val_loader, *_ = create_dataloaders(cfg)

Creating data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


In [16]:
batch_x, batch_stamp = next(iter(train_loader))
#iter(train_loader) DataLoader ka iterator banata hai aur next() us iterator se pehla batch retrieve karta hai.
#iss batch ko input (batch_x) aur target (batch_stamp) me unpack kiya jata hai

In [17]:
print(batch_x.shape)
print(batch_stamp.shape)

torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [18]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-base")

In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
batch_x = batch_x.to(device)

In [20]:
with torch.no_grad():
    (
        (reconstruction, encoder_output),    #Tokenizer input ko kitna accurately reconstruct kar pa raha hai & Encoder se nikle continuous features
        bsq_loss,                            #Tokenizer ki quantization/reconstruction training loss
        quantized,                           #Continuous features ka quantized (discrete) representation
        token_ids                            #Final discrete token IDs jo Kronos Transformer consume karega
    ) = tokenizer(batch_x)                   #Tokenizer continuous stock sequences ko discrete token IDs me convert karta hai

In [21]:
print(reconstruction.shape)
print(encoder_output.shape)
print(quantized.shape)
print(token_ids.shape)
print(bsq_loss)

torch.Size([32, 36, 6])
torch.Size([32, 36, 6])
torch.Size([32, 36, 20])
torch.Size([32, 36])
tensor(-0.0674)


In [22]:
print(token_ids.min())
print(token_ids.max())
print(token_ids.unique().numel()) #numel--Tensor me total elements count karta hai

tensor(13741)
tensor(1024423)
338


In [23]:
print(token_ids[:2])

tensor([[  30645,   30661,   32733,   30685,   30669,  278479,  691912,  998363,
          278415,  474079,  278415,  471979,  276399,  278415,  474031,  471979,
          459179, 1018179,  473995,  471983,  985483, 1018115,  888146, 1019219,
          675416,  998363, 1018115, 1016227,  887075,  887106,  886016,  839272,
          885828,  885792,  888146,  885074],
        [1019339,  887123,  885010, 1018251,  884994, 1018115,  998299, 1018131,
          887042,  885760,  885798, 1018243, 1018179,  887874,  885026,  884738,
         1018131,  677582,  669268,   30621,  816863,  474079,  278415,  461259,
         1018275,  278415,   30605,   30613,   30645,   16269,  278447,   30605,
           30597,  474027,  278415,   30637]])


In [24]:
print("Tokenizer Loss :", bsq_loss.item())

Tokenizer Loss : -0.0674005001783371


In [25]:
decoded = tokenizer.decode(token_ids)
print(decoded.shape)

torch.Size([32, 36, 6])


In [26]:
mse = (( batch_x.cpu()-decoded.cpu())**2).mean()
print(mse.item())

0.036786999553442


In [27]:
#does data loading, model initialization, optimization, validation, checkpoint saving, and sequential training phases
trainer = SequentialTrainer(config_path)
print(trainer)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [28]:
methods = [m for m in dir(trainer) if "train" in m.lower()]
print(methods)

['run_training', 'train_basemodel_phase', 'train_tokenizer_phase']


In [29]:
print(cfg)
print(cfg.batch_size)
print(cfg.lookback_window)
print(cfg.predict_window)

32
30
5


In [30]:
cfg.tokenizer_learning_rate = 2e-4
cfg.predictor_learning_rate = 4e-5

In [31]:
#Kronos Tokenizer ko fine-tune karta hai.Taaki tokenizer stock market data ko better discrete tokens me convert kar sake.
trainer.train_tokenizer_phase()

2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - === Tokenizer Training Started ===
INFO:tokenizer_training_rank_0:=== Tokenizer Training Started ===
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Experiment Name: PRISM
INFO:tokenizer_training_rank_0:Experiment Name: PRISM
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:tokenizer_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Rank: 0
INFO:tokenizer_training_rank_0:Rank: 0
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Timestamp: 2026-07-18 18:06:01
INFO:tokenizer_training_rank_0:Timestamp: 2026-07-18 18:06:01
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Loading pretrained tokenizer...
INFO:tokenizer_training_rank_0:Loading pretrained tokenizer...



Starting Tokenizer Fine-tuning Phase
Tokenizer model exists: True
Basemodel model exists: True
Loading pretrained tokenizer...


2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Tokenizer parameters: 3,958,042
INFO:tokenizer_training_rank_0:Tokenizer parameters: 3,958,042
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - === Training Configuration ===
INFO:tokenizer_training_rank_0:=== Training Configuration ===
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/prism_train.csv
INFO:tokenizer_training_rank_0:Data path: /content/Kronos/finetune_csv/data/prism_train.csv
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Lookback window: 30
INFO:tokenizer_training_rank_0:Lookback window: 30
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Predict window: 5
INFO:tokenizer_training_rank_0:Predict window: 5
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Batch size: 32
INFO:tokenizer_training_rank_0:Batch size: 32
2026-07-18 18:06:01 - tokenizer_training_rank_0 - INFO - Learning rate: 0.0002
INFO:tokenizer_training_rank_0:Learning

Tokenizer parameters: 3,958,042
Starting tokenizer fine-tuning training...
Creating tokenizer training data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


2026-07-18 18:07:03 - tokenizer_training_rank_0 - INFO - 
--- Epoch 1/2 Summary ---
Validation Loss: 0.0252
Epoch Time: 0:00:58
Total Training Time: 0:00:58

INFO:tokenizer_training_rank_0:
--- Epoch 1/2 Summary ---
Validation Loss: 0.0252
Epoch Time: 0:00:58
Total Training Time: 0:00:58

2026-07-18 18:07:03 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0252)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0252)


...

2026-07-18 18:07:36 - tokenizer_training_rank_0 - INFO - [Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
INFO:tokenizer_training_rank_0:[Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
2026-07-18 18:07:36 - tokenizer_training_rank_0 - INFO -   - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193
INFO:tokenizer_training_rank_0:  - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193


[Epoch 2/2, Step 24/26] LR: 0.000000, Loss: -0.0079
  - VQ Loss: -0.0674
  - Recon Loss Pre: 0.0324
  - Recon Loss All: 0.0193


2026-07-18 18:07:41 - tokenizer_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:37
Total Training Time: 0:00:37

INFO:tokenizer_training_rank_0:
--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:37
Total Training Time: 0:00:37

2026-07-18 18:07:41 - tokenizer_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)
INFO:tokenizer_training_rank_0:Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)
2026-07-18 18:07:41 - tokenizer_training_rank_0 - INFO - Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.66 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer
INFO:tokenizer_training_rank_0:Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.66 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer



--- Epoch 2/2 Summary ---
Validation Loss: 0.0242
Epoch Time: 0:00:37
Total Training Time: 0:00:37

Best model saved to: /content/Kronos/checkpoints/tokenizer/best_model (validation loss: 0.0242)

Tokenizer training completed! Best validation loss: 0.0242
Training time: 1.66 minutes
Model saved to: /content/Kronos/checkpoints/tokenizer


True

In [32]:
print("Tokenizer save path :", cfg.tokenizer_save_path)
print("Basemodel save path :", cfg.basemodel_save_path)
print("Base save path :", cfg.base_save_path)

Tokenizer save path : /content/Kronos/checkpoints/tokenizer
Basemodel save path : /content/Kronos/checkpoints/basemodel
Base save path : /content/Kronos/checkpoints


In [33]:
print("Tokenizer dir exists:", os.path.exists(cfg.tokenizer_save_path))
print("Basemodel dir exists:", os.path.exists(cfg.basemodel_save_path))

if os.path.exists(cfg.tokenizer_save_path):
    print(os.listdir(cfg.tokenizer_save_path))

Tokenizer dir exists: True
Basemodel dir exists: True
['best_model']


In [34]:
config["model_paths"]["finetuned_tokenizer"] = "/content/Kronos/checkpoints/tokenizer/best_model"
config["model_paths"]["tokenizer_save_name"] = "tokenizer"
config["model_paths"]["basemodel_save_name"] = "basemodel"

In [35]:
with open(config_path, "w") as f:
    yaml.dump(config, f)
cfg=CustomFinetuneConfig(config_path)

In [36]:
print("finetuned_tokenizer_path :", cfg.finetuned_tokenizer_path)
print("tokenizer_save_path      :", cfg.tokenizer_save_path)
print("basemodel_save_path      :", cfg.basemodel_save_path)

finetuned_tokenizer_path : /content/Kronos/checkpoints/tokenizer/best_model
tokenizer_save_path      : /content/Kronos/checkpoints/tokenizer
basemodel_save_path      : /content/Kronos/checkpoints/basemodel


In [37]:
trainer = SequentialTrainer(config_path)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [38]:
#Fine-tuned tokenizer ko use karke Kronos prediction model train karta hai.Taaki model stock returns ko accurately predict karna seekhe.
trainer.train_basemodel_phase()

2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - === Basemodel Training Started ===
INFO:basemodel_training_rank_0:=== Basemodel Training Started ===
2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Experiment Name: PRISM
INFO:basemodel_training_rank_0:Experiment Name: PRISM
2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Log Directory: /content/Kronos/checkpoints/logs
INFO:basemodel_training_rank_0:Log Directory: /content/Kronos/checkpoints/logs
2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Rank: 0
INFO:basemodel_training_rank_0:Rank: 0
2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Timestamp: 2026-07-18 18:07:41
INFO:basemodel_training_rank_0:Timestamp: 2026-07-18 18:07:41
2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Loading fine-tuned tokenizer...
INFO:basemodel_training_rank_0:Loading fine-tuned tokenizer...



Starting Basemodel Fine-tuning Phase
Tokenizer model exists: True
Basemodel model exists: True
Loading fine-tuned tokenizer...


2026-07-18 18:07:41 - basemodel_training_rank_0 - INFO - Loading pretrained predictor...
INFO:basemodel_training_rank_0:Loading pretrained predictor...


Loading weights from local directory
Loading pretrained predictor...


2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Model parameters: 102,310,592
INFO:basemodel_training_rank_0:Model parameters: 102,310,592
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - === Training Configuration ===
INFO:basemodel_training_rank_0:=== Training Configuration ===
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Data path: /content/Kronos/finetune_csv/data/prism_train.csv
INFO:basemodel_training_rank_0:Data path: /content/Kronos/finetune_csv/data/prism_train.csv
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Lookback window: 30
INFO:basemodel_training_rank_0:Lookback window: 30
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Predict window: 5
INFO:basemodel_training_rank_0:Predict window: 5
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Batch size: 32
INFO:basemodel_training_rank_0:Batch size: 32
2026-07-18 18:07:43 - basemodel_training_rank_0 - INFO - Learning rate: 4e-05
INFO:basemodel_training_rank_0:Learning rate

...

2026-07-18 18:28:01 - basemodel_training_rank_0 - INFO - 
--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 766.73 seconds

INFO:basemodel_training_rank_0:
--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 766.73 seconds




--- Epoch 2/2 Summary ---
Training Loss: 3.2586
Validation Loss: 3.7007
Epoch Time: 766.73 seconds



2026-07-18 18:28:08 - basemodel_training_rank_0 - INFO - Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)
INFO:basemodel_training_rank_0:Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)
2026-07-18 18:28:08 - basemodel_training_rank_0 - INFO - Basemodel training completed! Best validation loss: 3.7007
Training time: 20.41 minutes
Model saved to: /content/Kronos/checkpoints/basemodel
INFO:basemodel_training_rank_0:Basemodel training completed! Best validation loss: 3.7007
Training time: 20.41 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


Best model saved to: /content/Kronos/checkpoints/basemodel/best_model (validation loss: 3.7007)

Basemodel training completed! Best validation loss: 3.7007
Training time: 20.41 minutes
Model saved to: /content/Kronos/checkpoints/basemodel


True

In [39]:
import inspect
source = inspect.getsource(SequentialTrainer.train_basemodel_phase)
print(source)

    def train_basemodel_phase(self):
        print("\n" + "="*60)
        print("Starting Basemodel Fine-tuning Phase")
        print("="*60)
        
        if getattr(self.config, 'pre_trained_tokenizer', True):
            if not os.path.exists(self.config.finetuned_tokenizer_path):
                raise FileNotFoundError(f"Fine-tuned tokenizer does not exist: {self.config.finetuned_tokenizer_path}")
        
        _, basemodel_exists = self._check_existing_models()
        if basemodel_exists and self.config.skip_existing:
            print("Basemodel model already exists, skipping training")
            return True
        
        log_dir = os.path.join(self.config.base_save_path, "logs")
        logger = setup_basemodel_logging(self.config.exp_name, log_dir, self.rank)
        
        set_seed(self.config.seed)
        
        if getattr(self.config, 'pre_trained_tokenizer', True):
            logger.info("Loading fine-tuned tokenizer...")
            if self.rank == 0:
   

In [40]:
print("Tokenizer:")
print(os.listdir("/content/Kronos/checkpoints/tokenizer"))
print("Basemodel:")
print(os.listdir("/content/Kronos/checkpoints/basemodel"))

Tokenizer:
['best_model']
Basemodel:
['best_model']


In [41]:
print(os.path.exists("/content/Kronos/checkpoints/tokenizer/best_model"))
print(os.path.exists("/content/Kronos/checkpoints/basemodel/best_model"))

True
True


In [42]:
drive_model_dir = os.path.join( PROJECT_ROOT, "models", "kronos")
os.makedirs(drive_model_dir, exist_ok=True)

#shutil.copytree() → Pura folder + uske andar ki saari files copy karta hai
shutil.copytree("/content/Kronos/checkpoints/tokenizer",os.path.join(drive_model_dir, "tokenizer"),dirs_exist_ok=True)
shutil.copytree("/content/Kronos/checkpoints/basemodel",os.path.join(drive_model_dir, "basemodel"),dirs_exist_ok=True)

print("Models copied successfully.")

Models copied successfully.


#LORA

In [43]:
#Raw stock data → Tokenizer → Token sequences → Kronos Transformer → Prediction (logits) → Loss → Update only LoRA adapters

In [44]:
from model.kronos import Kronos, KronosTokenizer
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = Kronos.from_pretrained("/content/Kronos/checkpoints/basemodel/best_model").to(device)
base_model.eval()

print(type(base_model))

Loading weights from local directory
<class 'model.kronos.Kronos'>


In [45]:
#Base model ke saare parameters freeze kardo
for p in base_model.parameters():
    p.requires_grad = False
print("Backbone Frozen!")

Backbone Frozen!


In [46]:
lora_config = LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none", #Effective Scale ≈alpha/r,alpha is LoRA updates ka scaling factor yeh decide karega ki kitni strength se update karega
    target_modules=["q_proj","k_proj","v_proj","out_proj",])
#Input ko query/key/value vectors me convert karta hai. jabki out proj --Attention ka final output next transformer layer ko bhejta hai

In [47]:
!pip install -U torchao

In [48]:
#get_peft_model() wraps the pretrained Kronos model with LoRA adapters according to the provided configuration, enabling parameter-efficient fine-tuning
lora_model = get_peft_model(base_model,lora_config)
lora_model.to(device)

print(type(lora_model))

<class 'peft.peft_model.PeftModel'>


In [49]:
lora_model.print_trainable_parameters()

trainable params: 1,384,448 || all params: 103,695,040 || trainable%: 1.3351


In [50]:
#Ye code check karta hai ki model ke kaunse parameters train honge aur unke names trainable list me store karta hai
trainable = []
# name is para ka naam while p is actual para
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        trainable.append(name)

print(len(trainable))
print(trainable[:20])

104
['base_model.model.transformer.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.out_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.out_proj.lora_B.default.weight', 'base_model.model.transformer.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.transformer.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.transformer.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.transformer.1.self_attn.k_proj.lora_B.default.weight', 'base_model.model.transformer.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.transformer.1.self_attn.

In [51]:
tokenizer = KronosTokenizer.from_pretrained(
    "/content/Kronos/checkpoints/tokenizer/best_model").to(device)
tokenizer.eval()

print("Tokenizer Loaded!")

Loading weights from local directory
Tokenizer Loaded!


In [52]:
from torch.optim import AdamW
optimizer = AdamW(filter(lambda p: p.requires_grad, lora_model.parameters()),lr=2e-4,weight_decay=1e-2)
#it optimizes onlu those whose grad is updated
print("Optimizer Ready")

Optimizer Ready


In [53]:
print(lora_model.base_model.model.head.compute_loss)

<bound method DualHead.compute_loss of DualHead(
  (proj_s1): Linear(in_features=832, out_features=1024, bias=True)
  (proj_s2): Linear(in_features=832, out_features=1024, bias=True)
)>


In [54]:
batch_x, batch_stamp = next(iter(train_loader))
batch_x = batch_x.to(device)
batch_stamp = batch_stamp.to(device)

#batch_x ko trained tokenizer use karke do token sequences (token_seq_0 aur token_seq_1) me convert kar rahe hain.
#torch.no_grad() isliye use kiya hai kyunki tokenizer ko train nahi karna, sirf tokens generate karne hain.
with torch.no_grad():
    token_seq_0, token_seq_1 = tokenizer.encode(batch_x,half=True)

In [55]:
#Kronos dual-stream prediction karta hai.
#Wo sequence ke dono halves ko alag process karta hai aur dono ke liye logits predict karta hai.
print(token_seq_0.shape)
print(token_seq_1.shape)

torch.Size([32, 36])
torch.Size([32, 36])


In [56]:
#Ye LoRA-wrapped Kronos Transformer ko dono token seq aur timestamp features dekar dono streams ke prediction scores (s1_logits aur s2_logits) nikal raha hai.
s1_logits, s2_logits = lora_model.base_model.model(token_seq_0,token_seq_1,batch_stamp)

print(s1_logits.shape)
print(s2_logits.shape)

torch.Size([32, 36, 1024])
torch.Size([32, 36, 1024])


In [57]:
token_in_0 = token_seq_0[:, :-1]   #Last token hata kar model ka input bana rahe hain
token_in_1 = token_seq_1[:, :-1]

token_out_0 = token_seq_0[:, 1:]   #First token hata kar target bana rahe hain jise model predict karega
token_out_1 = token_seq_1[:, 1:]

stamp = batch_stamp[:, :-1]        #last timestamp remove
s1_logits, s2_logits = lora_model.base_model.model(token_in_0,token_in_1,stamp)

loss, s1_loss, s2_loss = (         #Predicted logits ko actual target tokens se compare karke total loss aur dono streams ka alag-alag loss calculate kar rahe hain
    lora_model.base_model.model.head.compute_loss(s1_logits,s2_logits,token_out_0,token_out_1))

#.item() → Tensor ko normal Python number bana do
print(loss.item())
print(s1_loss.item())
print(s2_loss.item())

3.169903516769409
3.730276107788086
2.6095309257507324


In [58]:
optimizer.zero_grad()  #Pichle iteration ke gradients ko clear karta hai taaki naye gradients unke upar add na ho
loss.backward()        #Loss ke basis par backpropagation chala kar har trainable parameter (LoRA weights) ka gradient calculate karta hai
optimizer.step()       #Calculated gradients use karke LoRA weights ko update karta hai, jisse model ki prediction improve hoti hai

print("One LoRA optimization step completed.")

One LoRA optimization step completed.


In [59]:
#Ye har trainable LoRA parameter ka average gradient print karta hai, taaki pata chale ki training ke time us parameter me update aa raha hai ya nahi
for name, p in lora_model.named_parameters():
    if p.requires_grad and p.grad is not None:    #irf trainable (LoRA) parameters ko consider karo .Check karo ki loss.backward() ke baad gradient calculate hua hai ya nahi
        print(name, p.grad.abs().mean().item())

base_model.model.transformer.0.self_attn.q_proj.lora_A.default.weight 0.0
base_model.model.transformer.0.self_attn.q_proj.lora_B.default.weight 6.1078239923517685e-06
base_model.model.transformer.0.self_attn.k_proj.lora_A.default.weight 0.0
base_model.model.transformer.0.self_attn.k_proj.lora_B.default.weight 5.263105322228512e-06
base_model.model.transformer.0.self_attn.v_proj.lora_A.default.weight 0.0
base_model.model.transformer.0.self_attn.v_proj.lora_B.default.weight 1.0980078513966873e-05
base_model.model.transformer.0.self_attn.out_proj.lora_A.default.weight 0.0
base_model.model.transformer.0.self_attn.out_proj.lora_B.default.weight 1.5858637198107317e-05
base_model.model.transformer.1.self_attn.q_proj.lora_A.default.weight 0.0
base_model.model.transformer.1.self_attn.q_proj.lora_B.default.weight 7.6074034041084815e-06
base_model.model.transformer.1.self_attn.k_proj.lora_A.default.weight 0.0
base_model.model.transformer.1.self_attn.k_proj.lora_B.default.weight 7.600231583637651e

In [60]:
for name, p in lora_model.named_parameters():
    if p.requires_grad:
        print(name)
        print("Gradient:", p.grad.abs().mean().item())     #add :.7f to see it till 7 decimal places
        break

base_model.model.transformer.0.self_attn.q_proj.lora_A.default.weight
Gradient: 0.0


In [61]:
#Raw Stock Data ->Tokenizer ->Token Sequences -> Kronos Transformer -> Prediction (Logits) ->Loss Calculation -> Backpropagation ->Update LoRA Weights ->Print Loss

In [62]:
lora_model.train()
for step in range(10):
    optimizer.zero_grad()

    with torch.no_grad():
        token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

#Input tokens ko Kronos Transformer me dekar next-token prediction ke logits generate karta hai
    s1_logits, s2_logits = lora_model.base_model.model(
        token_seq_0[:, :-1],
        token_seq_1[:, :-1],
        batch_stamp[:, :-1])

#Predicted logits aur actual next tokens compare karke training loss calculate karta hai
    loss, _, _ = lora_model.base_model.model.head.compute_loss(
        s1_logits,
        s2_logits,
        token_seq_0[:, 1:],
        token_seq_1[:, 1:])

    loss.backward()
    optimizer.step()

    print(f"Step {step+1}: {loss.item():.4f}")

Step 1: 3.1998
Step 2: 3.1932
Step 3: 3.1960
Step 4: 3.1861
Step 5: 3.1936
Step 6: 3.1897
Step 7: 3.1786
Step 8: 3.1784
Step 9: 3.1749
Step 10: 3.1760


#Evaluation

In [63]:
from copy import deepcopy

# Zero-shot Kronos
zero_model = Kronos.from_pretrained("NeoQuasar/Kronos-base")
zero_model.eval()

# LoRA trained Kronos
lora_model.eval()

print("Zero-shot loaded")
print("LoRA loaded")

Zero-shot loaded
LoRA loaded


In [64]:
zero_preds = []
lora_preds = []
targets = []

with torch.no_grad():
    token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

# Original (zero-shot) Kronos model se predictions nikal rahe hain
    s1_zero, s2_zero = zero_model(token_seq_0[:, :-1],token_seq_1[:, :-1],batch_stamp[:, :-1] )

#LoRA fine-tuned Kronos model se predictions nikal rahe hain
    s1_lora, s2_lora = lora_model.base_model.model(token_seq_0[:, :-1],token_seq_1[:, :-1],batch_stamp[:, :-1])

    pred_zero = torch.argmax(s2_zero, dim=-1) #Zero-shot model ke logits me se highest score wala token prediction select kar rahe hain
    pred_lora = torch.argmax(s2_lora, dim=-1) #LoRA model ke logits me se final predicted token nikal rahe hain
    target = token_seq_1[:, 1:]               #Actual next tokens ko target bana rahe hain jisse predictions compare hongi

print(pred_zero.shape)
print(pred_lora.shape)
print(target.shape)

torch.Size([32, 35])
torch.Size([32, 35])
torch.Size([32, 35])


In [65]:
#Zero-shot aur LoRA model ki token prediction accuracy calculate karta hai by comparing predicted tokens with actual target tokens
zero_acc = (pred_zero == target).float().mean().item()
lora_acc = (pred_lora == target).float().mean().item()

print(f"Zero-shot Accuracy : {100*zero_acc:.2f}%")
print(f"LoRA Accuracy      : {100*lora_acc:.2f}%")

Zero-shot Accuracy : 23.48%
LoRA Accuracy      : 27.86%


In [66]:
#Ye code LoRA model ki predictions aur actual targets ke ranking order ko compare karke har sample ka RankIC nikalta hai aur unka average print karta hai
from scipy.stats import spearmanr #Spearman Rank Correlation (RankIC) calculate karne ka function import kar rahe hain
rankics = []

for i in range(pred_lora.size(0)):
    ic, _ = spearmanr(pred_lora[i].cpu().numpy(),target[i].cpu().numpy())   #spearmanr() NumPy arrays leta hai
#Prediction aur target ke ranks kitne similar hain, uska correlation (RankIC) calculate karta hai.

    if not np.isnan(ic): #Agar RankIC valid hai tabhi usse store karo
        rankics.append(ic)

print("Average RankIC :", np.mean(rankics))

Average RankIC : 0.525813789421804


In [76]:
save_path = "/content/Kronos/checkpoints/lora"
os.makedirs(save_path, exist_ok=True)
lora_model.save_pretrained(save_path)
print("LoRA adapters saved!")

LoRA adapters saved!


In [77]:
import shutil
drive_lora_dir = os.path.join(PROJECT_ROOT, "models", "kronos", "lora")
os.makedirs(drive_lora_dir, exist_ok=True)

shutil.copytree(save_path, drive_lora_dir, dirs_exist_ok=True)
print("LoRA adapters copied to Drive:", drive_lora_dir)

LoRA adapters copied to Drive: /content/drive/MyDrive/PRISM/models/kronos/lora


In [68]:
from peft import PeftModel
base_model = Kronos.from_pretrained("/content/Kronos/checkpoints/basemodel/best_model")
loaded_lora = PeftModel.from_pretrained(base_model,"/content/Kronos/checkpoints/lora")

loaded_lora.eval()
print("Reload successful")

Loading weights from local directory
Reload successful


In [69]:
# Evaluation Function
##Ye code poori validation dataset par LoRA model and Zero shot ko evaluate karta hai aur uski Token Accuracy aur Average RankIC calculate karta hai
def evaluate_model(model, tokenizer, val_loader, device, is_lora=False):
    model.eval()
    total_correct = 0
    total_tokens = 0
    rankics = []

    with torch.no_grad():
        for batch_x, batch_stamp in val_loader:
            batch_x = batch_x.to(device)
            batch_stamp = batch_stamp.to(device)

            # Encode continuous features
            token_seq_0, token_seq_1 = tokenizer.encode(batch_x, half=True)

            # Forward
            if is_lora:
                s1_logits, s2_logits = model.base_model.model(
                    token_seq_0[:, :-1],
                    token_seq_1[:, :-1],
                    batch_stamp[:, :-1])

            else:
                s1_logits, s2_logits = model(
                    token_seq_0[:, :-1],
                    token_seq_1[:, :-1],
                    batch_stamp[:, :-1])

            # Predictions
            pred = torch.argmax(s2_logits,dim=-1)      #dim=-1 ka matlab hota hai last dimension par operation perform kar
            target = token_seq_1[:, 1:]

            # Accuracy
            total_correct += (pred == target).sum().item()
            total_tokens += target.numel()

            #Rank IC
            for i in range(pred.size(0)):

                ic, _ = spearmanr(                      #_ means mujhe ye value mil rahi hai, lekin usse use nahi karna
                    pred[i].cpu().numpy(),
                    target[i].cpu().numpy()
                )

                if not np.isnan(ic):
                    rankics.append(ic)

    accuracy = 100 * total_correct / total_tokens
    rankic = np.mean(rankics)
    return accuracy, rankic

# Evaluate Zero-shot Kronos
zero_acc, zero_rankic = evaluate_model(zero_model,tokenizer,val_loader,device,is_lora=False)

# Evaluate LoRA Kronos
lora_acc, lora_rankic = evaluate_model( loaded_lora, tokenizer, val_loader, device, is_lora=True)

In [70]:
print("Zero-shot Accuracy :",zero_acc)
print("LoRA Accuracy :",lora_acc)
print()
print("Zero-shot RankIC :",zero_rankic)
print("LoRA RankIC :",lora_rankic)

Zero-shot Accuracy : 21.496598639455783
LoRA Accuracy : 21.99546485260771

Zero-shot RankIC : 0.4694828702993812
LoRA RankIC : 0.42829283208012253


In [71]:
results = pd.DataFrame({
    "Model":["Zero-shot Kronos","LoRA Fine-tuned Kronos"],
    "Accuracy":[zero_acc,lora_acc],
    "RankIC":[zero_rankic,lora_rankic]
})
results

,Model,Accuracy,RankIC
0,Zero-shot Kronos,21.496599,0.469483
1,LoRA Fine-tuned Kronos,21.995465,0.428293


In [72]:
#Deliverable 2 Completed !!